# Publish the physics model on Hugging Face

Merges the half-strength adapter into Qwen3.5-2B, checks the merged model, writes the model card, and uploads it.

**What gets published:** one full model (about 4 GB, float16, text only: Qwen3.5's image encoder is not included), its tokenizer and chat template, the LoRA adapter itself (`adapter/`, 125 MB), a model card with the test results, and the two result reports.

**Run on one account with a T4 GPU**, top to bottom. About 40 minutes. Every step saves to Drive, so after a disconnect just run the cells again; finished steps are skipped.

**You need a Hugging Face access token with write permission:** huggingface.co > Settings > Access Tokens > Create new token > type "Write". Keep it secret; never paste it into a cell's code.

The repository is created **private** first. Check the page, then make it public on the website (Settings > Change visibility).

## 1. Settings

In [ ]:
WORKDIR = "/content/drive/MyDrive/physics_eval"
REPO_NAME = "qwen3.5-2b-physics"          # the model's name on Hugging Face (your-username/REPO_NAME)
PRIVATE = True                            # create it private; make it public on the website after checking
LICENSE = "cc-by-nc-sa-4.0"               # see the note below before changing

BASE = "Qwen/Qwen3.5-2B"
ADAPTER = f"{WORKDIR}/train_runs/qwen-v1.2-lora/ckpt-00647-x50"   # half-strength adapter (lora_alpha 32)
OUT_DIR = f"{WORKDIR}/publish/{REPO_NAME}"
REF_RUN = "ft-qwen-v1.2-lora-s647-x50"   # the adapter's own evaluation, to compare the merged model against
SAVE_DTYPE = "float16"   # every evaluation ran in float16; bf16 would round away part of the small LoRA change

**About `LICENSE`.** Qwen3.5-2B is Apache-2.0. Most of the training data comes from MegaScience, which is **CC BY-NC-SA 4.0** (non-commercial, share-alike); the PhysicsEval training split is CC BY-SA 4.0. Whether a dataset's licence carries over to a model trained on it is legally unsettled. `cc-by-nc-sa-4.0` is the cautious choice. `apache-2.0` is also common in practice; the model card then still lists the data licences. This is not legal advice.

## 2. Mount Drive and check the files

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os, sys, json, shutil
sys.path.insert(0, WORKDIR)
import physeval, phystrain
assert physeval.VERSION >= "1.4" and phystrain.VERSION >= "1.4", "use physeval.py and phystrain.py 1.4"
cfg = json.load(open(f"{ADAPTER}/adapter_config.json"))
assert cfg["lora_alpha"] == 32 and cfg["r"] == 32, f"expected the half-strength adapter, got {cfg['lora_alpha']}/{cfg['r']}"
for f in ["results/summary_final_test.md", "results/summary_dev_all.md"]:
    assert os.path.exists(f"{WORKDIR}/{f}"), f"missing {f} (PhysicsFinalTest section 5)"
os.makedirs(OUT_DIR, exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv
!free -g | head -2

## 3. Install packages

In [ ]:
!pip -q uninstall -y torchao
!pip -q install -U transformers peft accelerate safetensors huggingface_hub
!pip -q install "antlr4-python3-runtime==4.11.1" chardet

## 4. Merge the adapter into the model (about 10 minutes, uses about 9 GB of RAM)
Done on the CPU in full precision (float32), then saved in **float16**, the precision of every evaluation. (A first version saved bf16: bf16 keeps only 8 bits of precision, so part of the adapter's small weight changes was rounded away and 9 of 26 check answers came out different. If an older bf16 merge is in `OUT_DIR`, this cell replaces it.) If the session crashes with "out of RAM", tell me.

In [ ]:
import torch, gc, glob
def saved_dtype():
    p = f"{OUT_DIR}/config.json"
    if not os.path.exists(p):
        return None
    c = json.load(open(p))
    return str(c.get("dtype") or c.get("torch_dtype") or (c.get("text_config") or {}).get("dtype"))
if saved_dtype() == SAVE_DTYPE:
    print("already merged in", SAVE_DTYPE, ":", OUT_DIR)
else:
    for f in glob.glob(f"{OUT_DIR}/*.safetensors") + glob.glob(f"{OUT_DIR}/*.index.json") + \
             [f"{OUT_DIR}/config.json", f"{OUT_DIR}/generation_config.json"]:
        if os.path.exists(f):
            os.remove(f)   # an older merge in another precision
    phystrain.import_peft()
    from peft import PeftModel
    tok, model = phystrain.load_base(BASE, "float32")      # text model on the CPU
    model = PeftModel.from_pretrained(model, ADAPTER).merge_and_unload()
    model = model.to(getattr(torch, SAVE_DTYPE))
    model.save_pretrained(OUT_DIR, safe_serialization=True)
    tok.save_pretrained(OUT_DIR)
    del model; gc.collect()
    print("merged model saved to", OUT_DIR, "in", saved_dtype())
os.makedirs(f"{OUT_DIR}/adapter", exist_ok=True)   # the adapter itself, for use with PEFT on top of the base model
for f in ["adapter_config.json", "adapter_model.safetensors"]:
    shutil.copy(f"{ADAPTER}/{f}", f"{OUT_DIR}/adapter/{f}")
!ls -la "{OUT_DIR}" "{OUT_DIR}/adapter"

## 5. Check the merged model (about 10 minutes, GPU)
Answers the first 2 UGPhysics development questions of every subject (26 questions) with the merged model, exactly like our evaluations, and compares with the adapter's own answers from before.

Expected with float16: nearly all 26 answers word-for-word identical (a few may differ from tiny rounding), and the same number correct. Send me the output.

In [ ]:
CHECK_RUN = f"merged-check-{SAVE_DTYPE}"
!python "{WORKDIR}/physeval.py" generate --workdir "{WORKDIR}" --run {CHECK_RUN} --model "{OUT_DIR}" --enable-thinking false \
    --benchmarks ugphysics --per-group "ugphysics=2" --max-new-tokens 4096 --batch-size 16
!python "{WORKDIR}/physeval.py" grade --workdir "{WORKDIR}" --run {CHECK_RUN}
def load(run, kind):
    return {r["uid"]: r for r in map(json.loads, open(f"{WORKDIR}/runs/{run}/{kind}_ugphysics.jsonl"))}
gm, gr = load(CHECK_RUN, "gen"), load(REF_RUN, "gen")
km, kr = load(CHECK_RUN, "graded"), load(REF_RUN, "graded")
common = [u for u in gm if u in gr and u in km and u in kr]
same = sum(gm[u]["completion"] == gr[u]["completion"] for u in common)
print(f"questions compared: {len(common)}")
print(f"correct: merged {sum(bool(km[u]['rule']) for u in common)} | adapter {sum(bool(kr[u]['rule']) for u in common)}")
print(f"finished with a boxed answer: merged {sum(bool(km[u].get('has_boxed')) for u in common)} | adapter {sum(bool(kr[u].get('has_boxed')) for u in common)}")
print(f"word-for-word identical answers: {same} of {len(common)}")

## 6. Log in to Hugging Face
Paste your **write** token into the box that appears.

In [ ]:
from huggingface_hub import notebook_login, HfApi
notebook_login()

## 7. Write the model card

In [ ]:
api = HfApi()
USER = api.whoami()["name"]
REPO_ID = f"{USER}/{REPO_NAME}"
if LICENSE == "cc-by-nc-sa-4.0":
    LICENSE_TEXT = ("Released under **CC BY-NC-SA 4.0** (non-commercial, share-alike), the most restrictive licence among "
                    "its sources: the base model Qwen3.5-2B is Apache-2.0, most of the training data comes from "
                    "MegaScience (CC BY-NC-SA 4.0), and the PhysicsEval training split is CC BY-SA 4.0.")
else:
    LICENSE_TEXT = (f"Released under **{LICENSE}**. The base model Qwen3.5-2B is Apache-2.0. Note that most of the training "
                    "data comes from MegaScience (CC BY-NC-SA 4.0) and the PhysicsEval training split is CC BY-SA 4.0; "
                    "check those terms before commercial use.")
CARD = r'''---
license: {{LICENSE}}
base_model: Qwen/Qwen3.5-2B
library_name: transformers
pipeline_tag: text-generation
language:
- en
tags:
- physics
- science
- qwen3.5
- lora
datasets:
- MegaScience/MegaScience
- IUTVanguard/PhysicsEval
---

# Qwen3.5-2B Physics (non-thinking)

A physics fine-tune of [Qwen3.5-2B](https://huggingface.co/Qwen/Qwen3.5-2B), trained on a free Google Colab T4. It solves undergraduate physics problems with short step-by-step reasoning and a boxed final answer.

On **held-out UGPhysics questions** it scores **18.9% vs 15.9%** for the base model in the same (non-thinking) mode, with answers about 2.7× shorter. It is weaker than the base model at hard competition maths (see [Limitations](#limitations)).

## Results

All numbers come from one harness: UGPhysics (English) with its official prompt and rule-based judge, greedy decoding, at most 4,096 new tokens, float16 on a Tesla T4. UGPhysics accuracy is weighted by subject size. Only the rule judge is used, not the official two-stage protocol with an LLM judge (MARJ), so these numbers are not directly comparable with the UGPhysics leaderboard.

### Final test: questions never used for any decision

UGPhysics English, the questions ranked 100 to 199 in each subject: 12 subjects, 1,142 questions (GeometricalOptics has none left).

| model | accuracy [95% CI] | mean answer length | hit the 4,096-token limit |
|---|---|---|---|
| Qwen3.5-2B, non-thinking | 15.9% [13.6, 18.5] | 2,720 tokens | 48.9% |
| **this model** | **18.9% [16.3, 21.7]** | 1,006 tokens | 6.9% |

Paired difference: **+3.0 points** [+0.3, +5.7] (paired bootstrap 95% interval). This model is right on 102 questions the base model gets wrong and wrong on 75 that the base model gets right (exact McNemar p = 0.050).

### Development questions (used to choose the checkpoint and the adapter strength)

UGPhysics English, the questions ranked 0 to 99 in each subject.

| model | questions | accuracy |
|---|---|---|
| LiquidAI/LFM2.5-1.2B-Thinking | 1,258 | 11.3% |
| Qwen3.5-2B, thinking mode | 672 | 0.0%* |
| Qwen3.5-2B, non-thinking | 1,008 | 14.5% |
| **this model** | 1,258 | **18.3%** |

On the 1,008 questions both answered, this model leads the non-thinking base model by +3.3 points (17.9% vs 14.5%, McNemar p = 0.0035).

\* With 4,096 tokens and greedy decoding, Qwen3.5-2B in thinking mode never finished thinking (74% of answers hit the limit, 26% were stopped as repetition loops), so it never gave a final answer. It may do much better with a far larger token budget and sampling, which was not tested.

### MATH-500 (regression check)

210 questions (30 per subject), same grader.

| model | accuracy [95% CI] |
|---|---|
| Qwen3.5-2B, non-thinking | 73.0% [67.3, 78.7] |
| this model | 63.5% [56.6, 69.9] |

The drop (−9.4 points, McNemar p = 0.044) is mostly on the hardest problems (Level 5: 49% → 26%).

Full reports: [`eval/summary_final_test.md`](eval/summary_final_test.md) and [`eval/summary_dev_all.md`](eval/summary_dev_all.md).

## How to use

Use the non-thinking chat format and greedy decoding, as in the evaluation.

```python
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

repo = "{{REPO_ID}}"
tok = AutoTokenizer.from_pretrained(repo)
model = AutoModelForCausalLM.from_pretrained(repo, dtype=torch.float16, device_map="auto")

messages = [
    {"role": "system", "content": "Please reason step by step, and put your final answer within \\boxed{}."},
    {"role": "user", "content": "A 2.0 kg block slides from rest down a frictionless incline of height 1.5 m. "
                                "What is its speed at the bottom? Take g = 9.8 m/s^2."},
]
prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
inputs = tok(prompt, return_tensors="pt").to(model.device)
out = model.generate(**inputs, max_new_tokens=4096, do_sample=False)
print(tok.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True))
```

The weights are stored in float16, the precision used for every evaluation. Loading them as bfloat16 also works but rounds part of the fine-tune's small weight changes away, so answers can differ slightly from the reported results. Qwen3.5 needs a recent `transformers` (`pip install -U transformers`).

**Adapter only:** the half-strength LoRA adapter is in [`adapter/`](adapter). To apply it to the original model with PEFT:

```python
from peft import PeftModel
base = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3.5-2B", dtype=torch.float16, device_map="auto")
model = PeftModel.from_pretrained(base, "{{REPO_ID}}", subfolder="adapter").merge_and_unload()
```

## Training

- **Data:** 29,500 step-by-step solutions built mainly from [MegaScience](https://huggingface.co/datasets/MegaScience/MegaScience) (textbook physics, physics questions from NaturalReasoning, Nemotron-Science physics multiple choice, and some maths), plus the [PhysicsEval](https://huggingface.co/datasets/IUTVanguard/PhysicsEval) training split. Every question that overlapped a UGPhysics, PhysicsEval-test or MATH-500 question was removed (same text, a shared 13-word run, TF-IDF similarity of 0.4 or more, or a reworded copy with the same numbers). A random 12,000 of them were used.
- **Format:** Qwen's non-thinking chat format. The reasoning is plain text, followed by `So the final answer is \boxed{...}`. The loss covers the answer only.
- **Method:** LoRA, rank 32, alpha 64, on the attention (`q_proj`, `k_proj`, `v_proj`, `o_proj`), Gated DeltaNet (`in_proj_qkv`, `in_proj_z`, `out_proj`) and MLP (`gate_proj`, `up_proj`, `down_proj`) layers: 31M trainable parameters. Learning rate 1e-4 with 5% warm-up and cosine decay, about 18 examples per step, 647 steps (1 epoch), fp16 with loss scaling. About 4.5 hours on one T4.
- **Half strength:** the adapter is applied at 0.5× (alpha 32) before merging. At full strength the answers became too short for hard problems (MATH-500 fell to 51.7% on 70 questions); half strength kept the physics gain.
- **Text only:** Qwen3.5-2B's vision encoder is not included.

## Limitations

- About 4 of every 5 undergraduate physics problems are still answered wrongly. Always check the answers.
- Worse than the base model on hard maths (see MATH-500 above).
- About 7% of answers still run past 4,096 tokens.
- English only. Evaluated only with greedy decoding.

## License

{{LICENSE_TEXT}}

## Acknowledgements

The Qwen team (Qwen3.5-2B), the UGPhysics authors (benchmark and grader), and the creators of MegaScience and PhysicsEval.
'''
CARD = CARD.replace("{{LICENSE}}", LICENSE).replace("{{REPO_ID}}", REPO_ID).replace("{{LICENSE_TEXT}}", LICENSE_TEXT)
open(f"{OUT_DIR}/README.md", "w", encoding="utf-8").write(CARD)
os.makedirs(f"{OUT_DIR}/eval", exist_ok=True)
for f in ["summary_final_test.md", "summary_dev_all.md"]:
    shutil.copy(f"{WORKDIR}/results/{f}", f"{OUT_DIR}/eval/{f}")
from IPython.display import Markdown, display
display(Markdown(CARD))

## 8. Upload (about 5 minutes)
Creates `your-username/REPO_NAME` (private if `PRIVATE = True`) and uploads everything in `OUT_DIR`. Running it again uploads only what changed.

In [ ]:
api.create_repo(REPO_ID, private=PRIVATE, exist_ok=True)
api.upload_folder(folder_path=OUT_DIR, repo_id=REPO_ID, commit_message="Qwen3.5-2B physics fine-tune: merged model, model card, results")
print("done: https://huggingface.co/" + REPO_ID)

## 9. Test the uploaded model (optional, GPU)
Loads it from Hugging Face exactly as other people will and asks one question.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
tok = AutoTokenizer.from_pretrained(REPO_ID)
model = AutoModelForCausalLM.from_pretrained(REPO_ID, dtype=torch.float16).to("cuda").eval()
messages = [{"role": "system", "content": "Please reason step by step, and put your final answer within \\boxed{}."},
            {"role": "user", "content": "A 2.0 kg block slides from rest down a frictionless incline of height 1.5 m. "
                                        "What is its speed at the bottom? Take g = 9.8 m/s^2."}]
prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
inputs = tok(prompt, return_tensors="pt").to("cuda")
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=1024, do_sample=False)
print(tok.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True))